<a href="https://colab.research.google.com/github/satyaudaybandaru/Gemma-3-4B-FunctionCall/blob/main/Evaluation_and_Publish.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install --upgrade unsloth transformers huggingface_hub trl

#### Below Test Cases are defined, Our model is not finetuned for Multi tool call, but incase if you finetuned for it keep those test cases

In [ ]:
import json
import re
import torch
import pandas as pd


SYSTEM_PROMPT = """
You are a helpful assistant.

When a tool is required, respond only with a tool call.
Do not answer from your own knowledge when a tool is needed.
"""

TOOLS = [
    {
        "name": "get_current_time",
        "description": "Returns current date and time for a timezone.",
        "parameters": {
            "type": "object",
            "properties": {
                "timezone": {"type": "string"}
            }
        }
    },

    {
        "name": "calculator",
        "description": "Evaluates mathematical expressions.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            }
        }
    },
    {
        "name": "weather_lookup",
        "description": "Returns current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"}
            }
        }
    },
    {
        "name": "currency_converter",
        "description": "Converts one currency into another.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string"},
                "to_currency": {"type": "string"}
            }
        }
    },
    {
        "name": "stock_price",
        "description": "Returns latest stock price.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {"type": "string"}
            }
        }
    }
]


EVAL_CASES = [

    # single tool
    {
        "prompt": "What time is it in India?",
        "expected_tools": ["get_current_time"],
        "tool_required": True,
    },
    {
        "prompt": "What's the weather in Hyderabad?",
        "expected_tools": ["weather_lookup"],
        "tool_required": True,
    },
    {
        "prompt": "Calculate 247 * 89",
        "expected_tools": ["calculator"],
        "tool_required": True,
    },
    {
        "prompt": "Convert 100 USD to INR",
        "expected_tools": ["currency_converter"],
        "tool_required": True,
    },



    # no tool
    {
        "prompt": "Explain recursion with an example.",
        "expected_tools": [],
        "tool_required": False,
    },
    {
        "prompt": "Write a short poem about rain.",
        "expected_tools": [],
        "tool_required": False,
    },
    {
        "prompt": "Who are you?",
        "expected_tools": [],
        "tool_required": False,
    },

    ## Remove below 2 and 3 if not trained for multi tool call

    # 2 tools
    {
        "prompt": "What time is it in Tokyo and what's the weather there?",
        "expected_tools": [
            "get_current_time",
            "weather_lookup"
        ],
        "tool_required": True,
    },
    {
        "prompt": "Convert 500 USD to INR and tell me the current time in India",
        "expected_tools": [
            "currency_converter",
            "get_current_time"
        ],
        "tool_required": True,
    },

    # 3 tools
    {
        "prompt": "What is the weather in Hyderabad, what time is it in India and convert 100 USD to INR",
        "expected_tools": [
            "weather_lookup",
            "get_current_time",
            "currency_converter"
        ],
        "tool_required": True,
    },
    {
        "prompt": "Tell me Tesla stock price, current time in New York and weather in New York",
        "expected_tools": [
            "stock_price",
            "get_current_time",
            "weather_lookup"
        ],
        "tool_required": True,
    },

    # edge cases
    {
        "prompt": "Tell me about weather forecasting models.",
        "expected_tools": [],
        "tool_required": False,
    },
    {
        "prompt": "How do currency conversion websites work?",
        "expected_tools": [],
        "tool_required": False,
    },
    {
        "prompt": "What stock market indicators are commonly used?",
        "expected_tools": [],
        "tool_required": False,
    },
]


def generate_response(model, tk, prompt):

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": SYSTEM_PROMPT,
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                }
            ],
        },
    ]

    inputs = tk.apply_chat_template(
        messages,
        tools=TOOLS,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            eos_token_id=tk.eos_token_id,
            pad_token_id=tk.eos_token_id,
        )

    return tk.batch_decode(
        outputs,
        skip_special_tokens=False,
    )[0]


def extract_tool_calls(text):
    blocks = re.findall(
        r"<tool_call>(.*?)</tool_call>",
        text,
        flags=re.DOTALL,
    )

    if not blocks:
        return {
            "has_tool_call": False,
            "valid_json": False,
            "tool_calls": [],
            "error": "tool_call block not found",
        }

    all_calls = []
    for block in blocks:
        payload = block.strip()
        try:
            parsed = json.loads(payload)
            if isinstance(parsed, list):
                all_calls.extend(parsed)
            elif isinstance(parsed, dict):
                all_calls.append(parsed)
        except Exception as e:
            return {
                "has_tool_call": True,
                "valid_json": False,
                "tool_calls": [],
                "error": str(e),
            }

    return {
        "has_tool_call": True,
        "valid_json": True,
        "tool_calls": all_calls,
        "error": None,
    }


def evaluate_checkpoint(model, tk):

    model.eval()

    rows = []

    tool_correct = 0

    format_correct = 0
    json_correct = 0

    false_positive = 0
    false_negative = 0

    multi_tool_correct = 0
    multi_tool_total = 0

    for case in EVAL_CASES:

        response = generate_response(
            model,
            tk,
            case["prompt"],
        )

        parsed = extract_tool_calls(response)

        predicted_tools = [
            x.get("name")
            for x in parsed["tool_calls"]
            if isinstance(x, dict)
        ]

        expected_tools = case["expected_tools"]

        tool_match = (
            sorted(predicted_tools)
            ==
            sorted(expected_tools)
        )

        if tool_match:
            tool_correct += 1

        if case["tool_required"]:

            if parsed["has_tool_call"]:
                format_correct += 1

            if parsed["valid_json"]:
                json_correct += 1

        else:

            if not parsed["has_tool_call"]:
                format_correct += 1

            json_correct += 1

        if (
            not case["tool_required"]
            and len(predicted_tools) > 0
        ):
            false_positive += 1

        if (
            case["tool_required"]
            and len(predicted_tools) == 0
        ):
            false_negative += 1

        if len(expected_tools) > 1:

            multi_tool_total += 1

            if tool_match:
                multi_tool_correct += 1

        rows.append(
            {
                "prompt": case["prompt"],
                "expected_tools": expected_tools,
                "predicted_tools": predicted_tools,
                "tool_required": case["tool_required"],
                "tool_match": tool_match,
                "json_valid": parsed["valid_json"],
                "json_error": parsed["error"],
                "model_output": response,
            }
        )

    total = len(EVAL_CASES)

    metrics = {

        "tool_accuracy":
            tool_correct / total,

        "format_accuracy":
            format_correct / total,

        "json_accuracy":
            json_correct / total,

        "false_positive_rate":
            false_positive / total,

        "false_negative_rate":
            false_negative / total,

        "multi_tool_accuracy":
            multi_tool_correct /
            max(1, multi_tool_total),
    }

    df = pd.DataFrame(rows)

    print("\n===== METRICS =====")

    for k, v in metrics.items():
        print(f"{k}: {v:.2%}")

    return metrics, df

In [ ]:
from transformers import AutoTokenizer
from unsloth import FastModel
from peft import PeftModel

# Add checkpoints that you want to evaluate
checkpoints = [
    "checkpoint-50",
    # "checkpoint-20",
    # "checkpoint-30",
]

for ckpt in checkpoints:

    path = f"/content/drive/MyDrive/gemma3-4b-stage2toolcall/{ckpt}"

    tk = AutoTokenizer.from_pretrained(path)

    base_model, _ = FastModel.from_pretrained(
        model_name="unsloth/gemma-3-4b-it",
        max_seq_length=4096,
        load_in_4bit=True,
    )

    model = PeftModel.from_pretrained(
        base_model,
        path
    )

    print(f"\n===== {ckpt} =====")

    metrics, _ = evaluate_checkpoint(
        model,
        tk
    )

#### Create repo in hugging face
#### Replace `your_id`, `repo_id`, insert `token` and run the below cell to upload

In [ ]:
from huggingface_hub import upload_folder


upload_folder(
    folder_path="/content/drive/MyDrive/gemma3-4b-stage2toolcall/checkpoint-50",   # This is your local folder name
    repo_id="your_id/repo_id",# CHANGE HERE
    commit_message="Uploading adapter and modified tokenizer",
    token = "YOUR_HF_TOKEN" # INSERT TOKEN HERE"
)


## 🎉 That's It — Cheers!

* **Model Link:** [**Hugging Face Model**](https://huggingface.co/SatyaUdayB/gemma-3-4b-FC)

Thank you for checking out my work! This is my very first fine tuning project, and I truly appreciate your time and interest in exploring it.

If you have any suggestions, feedback, or ideas for improvement, feel free to reach out, I'd love to hear from you! 🚀

### Connect with Me

* [**LinkedIn**](https://linkedin.com/in/satyaudaybandaru)
* [**X (Twitter)**](https://x.com/SatyaUdayB)

Thanks again for your support and happy building! ✨
